# Candidate ReRank Model using Handcrafted Rules
In this notebook, we present a "candidate rerank" model using handcrafted rules. We can improve this model by engineering features, merging them unto items and users, and training a reranker model (such as XGB) to choose our final 20. Furthermore to tune and improve this notebook, we should build a local CV scheme to experiment new logic and/or models.

UPDATE: I published a notebook to compute validation score [here][10] using Radek's scheme described [here][11].

Note in this competition, a "session" actually means a unique "user". So our task is to predict what each of the `1,671,803` test "users" (i.e. "sessions") will do in the future. For each test "user" (i.e. "session") we must predict what they will `click`, `cart`, and `order` during the remainder of the week long test period.

### Step 1 - Generate Candidates
For each test user, we generate possible choices, i.e. candidates. In this notebook, we generate candidates from 5 sources:
* User history of clicks, carts, orders
* Most popular 20 clicks, carts, orders during test week
* Co-visitation matrix of click/cart/order to cart/order with type weighting
* Co-visitation matrix of cart/order to cart/order called buy2buy
* Co-visitation matrix of click/cart/order to clicks with time weighting

### Step 2 - ReRank and Choose 20
Given the list of candidates, we must select 20 to be our predictions. In this notebook, we do this with a set of handcrafted rules. We can improve our predictions by training an XGBoost model to select for us. Our handcrafted rules give priority to:
* Most recent previously visited items
* Items previously visited multiple times
* Items previously in cart or order
* Co-visitation matrix of cart/order to cart/order
* Current popular items

![](https://raw.githubusercontent.com/cdeotte/Kaggle_Images/main/Nov-2022/c_r_model.png)
  
# Credits
We thank many Kagglers who have shared ideas. We use co-visitation matrix idea from Vladimir [here][1]. We use groupby sort logic from Sinan in comment section [here][4]. We use duplicate prediction removal logic from Radek [here][5]. We use multiple visit logic from Pietro [here][2]. We use type weighting logic from Ingvaras [here][3]. We use leaky test data from my previous notebook [here][4]. And some ideas may have originated from Tawara [here][6] and KJ [here][7]. We use Colum2131's parquets [here][8]. Above image is from Ravi's discussion about candidate rerank models [here][9]

[1]: https://www.kaggle.com/code/vslaykovsky/co-visitation-matrix
[2]: https://www.kaggle.com/code/pietromaldini1/multiple-clicks-vs-latest-items
[3]: https://www.kaggle.com/code/ingvarasgalinskas/item-type-vs-multiple-clicks-vs-latest-items
[4]: https://www.kaggle.com/code/cdeotte/test-data-leak-lb-boost
[5]: https://www.kaggle.com/code/radek1/co-visitation-matrix-simplified-imprvd-logic
[6]: https://www.kaggle.com/code/ttahara/otto-mors-aid-frequency-baseline
[7]: https://www.kaggle.com/code/whitelily/co-occurrence-baseline
[8]: https://www.kaggle.com/datasets/columbia2131/otto-chunk-data-inparquet-format
[9]: https://www.kaggle.com/competitions/otto-recommender-system/discussion/364721
[10]: https://www.kaggle.com/cdeotte/compute-validation-score-cv-564
[11]: https://www.kaggle.com/competitions/otto-recommender-system/discussion/364991

# Notes
Below are notes about versions:
* **Version 1 LB 0.573** Uses popular ideas from public notebooks and adds additional co-visitation matrices and additional logic. Has CV `0.563`. See validation notebook version 2 [here][1].
* **Version 2 LB 573** Refactor logic for `suggest_buys(df)` to make it clear how new co-visitation matrices are reranking the candidates by adding to candidate weights. Also new logic boosts CV by `+0.0003`. Also LB is slightly better too. See validation notebook version 3 [here][1]
* **Version 3** is the same as version 2 but 1.5x faster co-visitation matrix computation!
* **Version 4 LB 575** Use top20 for clicks and top15 for carts and buys (instead of top40 and top40). This boosts CV `+0.0015` hooray! New CV is `0.5647`. See validation version 5 [here][1]
* **Version 5** is the same as version 4 but 2x faster co-visitation matrix computation! (and 3x faster than version 1)
* **Version 6** Stay tuned for more versions...

[1]: https://www.kaggle.com/code/cdeotte/compute-validation-score-cv-564

# Step 1 - Candidate Generation with RAPIDS
For candidate generation, we build three co-visitation matrices. One computes the popularity of cart/order given a user's previous click/cart/order. We apply type weighting to this matrix. One computes the popularity of cart/order given a user's previous cart/order. We call this "buy2buy" matrix. One computes the popularity of clicks given a user previously click/cart/order.  We apply time weighting to this matrix. We will use RAPIDS cuDF GPU to compute these matrices quickly!

In [3]:
VER = 5

import pandas as pd, numpy as np
from tqdm.notebook import tqdm
import os, sys, pickle, glob, gc
from collections import Counter
import cudf, itertools
print('We will use RAPIDS version',cudf.__version__)

We will use RAPIDS version 21.10.01


## Compute Three Co-visitation Matrices with RAPIDS
We will compute 3 co-visitation matrices using RAPIDS cuDF on GPU. This is 30x faster than using Pandas CPU like other public notebooks! For maximum speed, set the variable `DISK_PIECES` to the smallest number possible based on the GPU you are using without incurring memory errors. If you run this code offline with 32GB GPU ram, then you can use `DISK_PIECES = 1` and compute each co-visitation matrix in almost 1 minute! Kaggle's GPU only has 16GB ram, so we use `DISK_PIECES = 4` and it takes an amazing 3 minutes each! Below are some of the tricks to speed up computation
* Use RAPIDS cuDF GPU instead of Pandas CPU
* Read disk once and save in CPU RAM for later GPU multiple use
* Process largest amount of data possible on GPU at one time
* Merge data in two stages. Multiple small to single medium. Multiple medium to single large.
* Write result as parquet instead of dictionary

In [4]:
%%time
# CACHE FUNCTIONS
def read_file(f):
    return cudf.DataFrame( data_cache[f] )
def read_file_to_cache(f):
    df = pd.read_parquet(f)
    df.ts = (df.ts/1000).astype('int32')
    df['type'] = df['type'].map(type_labels).astype('int8')
    return df

# CACHE THE DATA ON CPU BEFORE PROCESSING ON GPU
data_cache = {}
type_labels = {'clicks':0, 'carts':1, 'orders':2}
files = glob.glob('../input/otto-chunk-data-inparquet-format/*_parquet/*')
for f in files: data_cache[f] = read_file_to_cache(f)

# CHUNK PARAMETERS
READ_CT = 5
CHUNK = int( np.ceil( len(files)/6 ))
print(f'We will process {len(files)} files, in groups of {READ_CT} and chunks of {CHUNK}.')

We will process 146 files, in groups of 5 and chunks of 25.
CPU times: user 47.2 s, sys: 11.7 s, total: 59 s
Wall time: 49.1 s


## 1) "Carts Orders" Co-visitation Matrix - Type Weighted

In [5]:
%%time
type_weight = {0:1, 1:6, 2:3}

# USE SMALLEST DISK_PIECES POSSIBLE WITHOUT MEMORY ERROR
DISK_PIECES = 4
SIZE = 1.86e6/DISK_PIECES

# COMPUTE IN PARTS FOR MEMORY MANGEMENT
for PART in range(DISK_PIECES):
    print()
    print('### DISK PART',PART+1)
    
    # MERGE IS FASTEST PROCESSING CHUNKS WITHIN CHUNKS
    # => OUTER CHUNKS
    for j in range(6):
        a = j*CHUNK
        b = min( (j+1)*CHUNK, len(files) )
        print(f'Processing files {a} thru {b-1} in groups of {READ_CT}...')
        
        # => INNER CHUNKS
        for k in range(a,b,READ_CT):
            # READ FILE
            df = [read_file(files[k])]
            for i in range(1,READ_CT): 
                if k+i<b: df.append( read_file(files[k+i]) )
            df = cudf.concat(df,ignore_index=True,axis=0)
            df = df.sort_values(['session','ts'],ascending=[True,False])
            # USE TAIL OF SESSION
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df.n<30].drop('n',axis=1)
            # CREATE PAIRS
            df = df.merge(df,on='session')
            df = df.loc[ ((df.ts_x - df.ts_y).abs()< 24 * 60 * 60) & (df.aid_x != df.aid_y) ]
            # MEMORY MANAGEMENT COMPUTE IN PARTS
            df = df.loc[(df.aid_x >= PART*SIZE)&(df.aid_x < (PART+1)*SIZE)]
            # ASSIGN WEIGHTS
            df = df[['session', 'aid_x', 'aid_y','type_y']].drop_duplicates(['session', 'aid_x', 'aid_y'])
            df['wgt'] = df.type_y.map(type_weight)
            df = df[['aid_x','aid_y','wgt']]
            df.wgt = df.wgt.astype('float32')
            df = df.groupby(['aid_x','aid_y']).wgt.sum()
            # COMBINE INNER CHUNKS
            if k==a: tmp2 = df
            else: tmp2 = tmp2.add(df, fill_value=0)
            print(k,', ',end='')
        print()
        # COMBINE OUTER CHUNKS
        if a==0: tmp = tmp2
        else: tmp = tmp.add(tmp2, fill_value=0)
        del tmp2, df
        gc.collect()
    # CONVERT MATRIX TO DICTIONARY
    tmp = tmp.reset_index()
    tmp = tmp.sort_values(['aid_x','wgt'],ascending=[True,False])
    # SAVE TOP 40
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp.n<15].drop('n',axis=1)
    # SAVE PART TO DISK (convert to pandas first uses less memory)
    tmp.to_pandas().to_parquet(f'top_15_carts_orders_v{VER}_{PART}.pqt')


### DISK PART 1
Processing files 0 thru 24 in groups of 5...


/opt/conda/lib/python3.7/site-packages/cudf/core/frame.py:2600: UserWarning: When using a sequence of booleans for `ascending`, `na_position` flag is not yet supported and defaults to treating nulls as greater than all numbers
  "When using a sequence of booleans for `ascending`, "


0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 2
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 3
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 

## 2) "Buy2Buy" Co-visitation Matrix

In [6]:
%%time
# USE SMALLEST DISK_PIECES POSSIBLE WITHOUT MEMORY ERROR
DISK_PIECES = 1
SIZE = 1.86e6/DISK_PIECES

# COMPUTE IN PARTS FOR MEMORY MANGEMENT
for PART in range(DISK_PIECES):
    print()
    print('### DISK PART',PART+1)
    
    # MERGE IS FASTEST PROCESSING CHUNKS WITHIN CHUNKS
    # => OUTER CHUNKS
    for j in range(6):
        a = j*CHUNK
        b = min( (j+1)*CHUNK, len(files) )
        print(f'Processing files {a} thru {b-1} in groups of {READ_CT}...')
        
        # => INNER CHUNKS
        for k in range(a,b,READ_CT):
            # READ FILE
            df = [read_file(files[k])]
            for i in range(1,READ_CT): 
                if k+i<b: df.append( read_file(files[k+i]) )
            df = cudf.concat(df,ignore_index=True,axis=0)
            df = df.loc[df['type'].isin([1,2])] # ONLY WANT CARTS AND ORDERS
            df = df.sort_values(['session','ts'],ascending=[True,False])
            # USE TAIL OF SESSION
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df.n<30].drop('n',axis=1)
            # CREATE PAIRS
            df = df.merge(df,on='session')
            df = df.loc[ ((df.ts_x - df.ts_y).abs()< 14 * 24 * 60 * 60) & (df.aid_x != df.aid_y) ] # 14 DAYS
            # MEMORY MANAGEMENT COMPUTE IN PARTS
            df = df.loc[(df.aid_x >= PART*SIZE)&(df.aid_x < (PART+1)*SIZE)]
            # ASSIGN WEIGHTS
            df = df[['session', 'aid_x', 'aid_y','type_y']].drop_duplicates(['session', 'aid_x', 'aid_y'])
            df['wgt'] = 1
            df = df[['aid_x','aid_y','wgt']]
            df.wgt = df.wgt.astype('float32')
            df = df.groupby(['aid_x','aid_y']).wgt.sum()
            # COMBINE INNER CHUNKS
            if k==a: tmp2 = df
            else: tmp2 = tmp2.add(df, fill_value=0)
            print(k,', ',end='')
        print()
        # COMBINE OUTER CHUNKS
        if a==0: tmp = tmp2
        else: tmp = tmp.add(tmp2, fill_value=0)
        del tmp2, df
        gc.collect()
    # CONVERT MATRIX TO DICTIONARY
    tmp = tmp.reset_index()
    tmp = tmp.sort_values(['aid_x','wgt'],ascending=[True,False])
    # SAVE TOP 40
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp.n<15].drop('n',axis=1)
    # SAVE PART TO DISK (convert to pandas first uses less memory)
    tmp.to_pandas().to_parquet(f'top_15_buy2buy_v{VER}_{PART}.pqt')


### DISK PART 1
Processing files 0 thru 24 in groups of 5...
0 , 5 , 

/opt/conda/lib/python3.7/site-packages/cudf/core/frame.py:2600: UserWarning: When using a sequence of booleans for `ascending`, `na_position` flag is not yet supported and defaults to treating nulls as greater than all numbers
  "When using a sequence of booleans for `ascending`, "


10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 
CPU times: user 20.6 s, sys: 8.81 s, total: 29.4 s
Wall time: 29.8 s


## 3) "Clicks" Co-visitation Matrix - Time Weighted

In [7]:
%%time
# USE SMALLEST DISK_PIECES POSSIBLE WITHOUT MEMORY ERROR
DISK_PIECES = 4
SIZE = 1.86e6/DISK_PIECES

# COMPUTE IN PARTS FOR MEMORY MANGEMENT
for PART in range(DISK_PIECES):
    print()
    print('### DISK PART',PART+1)
    
    # MERGE IS FASTEST PROCESSING CHUNKS WITHIN CHUNKS
    # => OUTER CHUNKS
    for j in range(6):
        a = j*CHUNK
        b = min( (j+1)*CHUNK, len(files) )
        print(f'Processing files {a} thru {b-1} in groups of {READ_CT}...')
        
        # => INNER CHUNKS
        for k in range(a,b,READ_CT):
            # READ FILE
            df = [read_file(files[k])]
            for i in range(1,READ_CT): 
                if k+i<b: df.append( read_file(files[k+i]) )
            df = cudf.concat(df,ignore_index=True,axis=0)
            df = df.sort_values(['session','ts'],ascending=[True,False])
            # USE TAIL OF SESSION
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df.n<30].drop('n',axis=1)
            # CREATE PAIRS
            df = df.merge(df,on='session')
            df = df.loc[ ((df.ts_x - df.ts_y).abs()< 24 * 60 * 60) & (df.aid_x != df.aid_y) ]
            # MEMORY MANAGEMENT COMPUTE IN PARTS
            df = df.loc[(df.aid_x >= PART*SIZE)&(df.aid_x < (PART+1)*SIZE)]
            # ASSIGN WEIGHTS
            df = df[['session', 'aid_x', 'aid_y','ts_x']].drop_duplicates(['session', 'aid_x', 'aid_y'])
            df['wgt'] = 1 + 3*(df.ts_x - 1659304800)/(1662328791-1659304800)
            df = df[['aid_x','aid_y','wgt']]
            df.wgt = df.wgt.astype('float32')
            df = df.groupby(['aid_x','aid_y']).wgt.sum()
            # COMBINE INNER CHUNKS
            if k==a: tmp2 = df
            else: tmp2 = tmp2.add(df, fill_value=0)
            print(k,', ',end='')
        print()
        # COMBINE OUTER CHUNKS
        if a==0: tmp = tmp2
        else: tmp = tmp.add(tmp2, fill_value=0)
        del tmp2, df
        gc.collect()
    # CONVERT MATRIX TO DICTIONARY
    tmp = tmp.reset_index()
    tmp = tmp.sort_values(['aid_x','wgt'],ascending=[True,False])
    # SAVE TOP 40
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp.n<20].drop('n',axis=1)
    # SAVE PART TO DISK (convert to pandas first uses less memory)
    tmp.to_pandas().to_parquet(f'top_20_clicks_v{VER}_{PART}.pqt')


### DISK PART 1
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 2
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 20 , 
Processing files 25 thru 49 in groups of 5...
25 , 30 , 35 , 40 , 45 , 
Processing files 50 thru 74 in groups of 5...
50 , 55 , 60 , 65 , 70 , 
Processing files 75 thru 99 in groups of 5...
75 , 80 , 85 , 90 , 95 , 
Processing files 100 thru 124 in groups of 5...
100 , 105 , 110 , 115 , 120 , 
Processing files 125 thru 145 in groups of 5...
125 , 130 , 135 , 140 , 145 , 

### DISK PART 3
Processing files 0 thru 24 in groups of 5...
0 , 5 , 10 , 15 , 

In [8]:
# FREE MEMORY
del data_cache, tmp
_ = gc.collect()

# Step 2 - ReRank (choose 20) using handcrafted rules
For description of the handcrafted rules, read this notebook's intro.

In [9]:
def load_test():    
    dfs = []
    for e, chunk_file in enumerate(glob.glob('../input/otto-chunk-data-inparquet-format/test_parquet/*')):
        chunk = pd.read_parquet(chunk_file)
        chunk.ts = (chunk.ts/1000).astype('int32')
        chunk['type'] = chunk['type'].map(type_labels).astype('int8')
        dfs.append(chunk)
    return pd.concat(dfs).reset_index(drop=True) #.astype({"ts": "datetime64[ms]"})

test_df = load_test()
print('Test data has shape',test_df.shape)
test_df.head()

Test data has shape (6928123, 4)


,session,aid,ts,type
0,13099779,245308,1661795832,0
1,13099779,245308,1661795862,1
2,13099779,972319,1661795888,0
3,13099779,972319,1661795898,1
4,13099779,245308,1661795907,0


In [10]:
%%time
def pqt_to_dict(df):
    return df.groupby('aid_x').aid_y.apply(list).to_dict()
# LOAD THREE CO-VISITATION MATRICES
top_20_clicks = pqt_to_dict( pd.read_parquet(f'top_20_clicks_v{VER}_0.pqt') )
for k in range(1,DISK_PIECES): 
    top_20_clicks.update( pqt_to_dict( pd.read_parquet(f'top_20_clicks_v{VER}_{k}.pqt') ) )
top_20_buys = pqt_to_dict( pd.read_parquet(f'top_15_carts_orders_v{VER}_0.pqt') )
for k in range(1,DISK_PIECES): 
    top_20_buys.update( pqt_to_dict( pd.read_parquet(f'top_15_carts_orders_v{VER}_{k}.pqt') ) )
top_20_buy2buy = pqt_to_dict( pd.read_parquet(f'top_15_buy2buy_v{VER}_0.pqt') )

# TOP CLICKS AND ORDERS IN TEST
top_clicks = test_df.loc[test_df['type']=='clicks','aid'].value_counts().index.values[:20]
top_orders = test_df.loc[test_df['type']=='orders','aid'].value_counts().index.values[:20]

print('Here are size of our 3 co-visitation matrices:')
print( len( top_20_clicks ), len( top_20_buy2buy ), len( top_20_buys ) )

Here are size of our 3 co-visitation matrices:
1837166 1168768 1837166
CPU times: user 1min 19s, sys: 6.07 s, total: 1min 25s
Wall time: 1min 19s


In [11]:
#type_weight_multipliers = {'clicks': 1, 'carts': 6, 'orders': 3}
type_weight_multipliers = {0: 1, 1: 6, 2: 3}

def suggest_clicks(df):
    # USER HISTORY AIDS AND TYPES
    aids=df.aid.tolist()
    types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1] ))
    # RERANK CANDIDATES USING WEIGHTS
    if len(unique_aids)>=20:
        weights=np.logspace(0.1,1,len(aids),base=2, endpoint=True)-1
        aids_temp = Counter() 
        # RERANK BASED ON REPEAT ITEMS AND TYPE OF ITEMS
        for aid,w,t in zip(aids,weights,types): 
            aids_temp[aid] += w * type_weight_multipliers[t]
        sorted_aids = [k for k,v in aids_temp.most_common(20)]
        return sorted_aids
    # USE "CLICKS" CO-VISITATION MATRIX
    aids2 = list(itertools.chain(*[top_20_clicks[aid] for aid in unique_aids if aid in top_20_clicks]))
    # RERANK CANDIDATES
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2).most_common(20) if aid2 not in unique_aids]    
    result = unique_aids + top_aids2[:20 - len(unique_aids)]
    # USE TOP20 TEST CLICKS
    return result + list(top_clicks)[:20-len(result)]

def suggest_buys(df):
    # USER HISTORY AIDS AND TYPES
    aids=df.aid.tolist()
    types = df.type.tolist()
    # UNIQUE AIDS AND UNIQUE BUYS
    unique_aids = list(dict.fromkeys(aids[::-1] ))
    df = df.loc[(df['type']==1)|(df['type']==2)]
    unique_buys = list(dict.fromkeys( df.aid.tolist()[::-1] ))
    # RERANK CANDIDATES USING WEIGHTS
    if len(unique_aids)>=20:
        weights=np.logspace(0.5,1,len(aids),base=2, endpoint=True)-1
        aids_temp = Counter() 
        # RERANK BASED ON REPEAT ITEMS AND TYPE OF ITEMS
        for aid,w,t in zip(aids,weights,types): 
            aids_temp[aid] += w * type_weight_multipliers[t]
        # RERANK CANDIDATES USING "BUY2BUY" CO-VISITATION MATRIX
        aids3 = list(itertools.chain(*[top_20_buy2buy[aid] for aid in unique_buys if aid in top_20_buy2buy]))
        for aid in aids3: aids_temp[aid] += 0.1
        sorted_aids = [k for k,v in aids_temp.most_common(20)]
        return sorted_aids
    # USE "CART ORDER" CO-VISITATION MATRIX
    aids2 = list(itertools.chain(*[top_20_buys[aid] for aid in unique_aids if aid in top_20_buys]))
    # USE "BUY2BUY" CO-VISITATION MATRIX
    aids3 = list(itertools.chain(*[top_20_buy2buy[aid] for aid in unique_buys if aid in top_20_buy2buy]))
    # RERANK CANDIDATES
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2+aids3).most_common(20) if aid2 not in unique_aids] 
    result = unique_aids + top_aids2[:20 - len(unique_aids)]
    # USE TOP20 TEST ORDERS
    return result + list(top_orders)[:20-len(result)]

In [12]:
# —— 保存基线提交，避免被后续 Notebook 覆盖 ——
import shutil, glob, os, pandas as pd

# 1) 备份规则版提交
if os.path.exists("submission.csv"):
    shutil.copy("submission.csv", "submission_rules.csv")

# 2) 列出将要复用的共现矩阵产物（都在 /kaggle/working 下，会随版本提交成为 Outputs）
artifacts = sorted(glob.glob("top_20_clicks_v*.pqt")) \
         +  sorted(glob.glob("top_15_carts_orders_v*.pqt")) \
         +  sorted(glob.glob("top_15_buy2buy_v*.pqt"))
print("Artifacts to reuse:", len(artifacts))
pd.Series(artifacts).to_csv("artifact_list.txt", index=False, header=False)


Artifacts to reuse: 9


# XGboost

## Cell 1 — 备份当前规则提交 + 安装兼容版 XGBoost + 修正热门统计

In [22]:
# 备份当前规则提交，避免被覆盖
import os, shutil
if os.path.exists("submission.csv"):
    shutil.copy("submission.csv", "submission_rules.csv")
    print("Backed up baseline to submission_rules.csv")

# 安装与 Py3.7 兼容的 xgboost 版本
!pip -q install xgboost==1.6.2
import xgboost as xgb, numpy as np, pandas as pd
print("xgboost version:", xgb.__version__)

# 你的 test_df 里 'type' 已经映射成 int8(0/1/2)，
# 这里修正之前用字符串统计热门导致取空的问题
top_clicks = test_df.loc[test_df['type']==0, 'aid'].value_counts().index.values[:20]
top_orders = test_df.loc[test_df['type']==2, 'aid'].value_counts().index.values[:20]
top_clicks_set, top_orders_set = set(top_clicks), set(top_orders)
print("Top clicks/orders ready.", len(top_clicks), len(top_orders))


xgboost version: 1.6.2
Top clicks/orders ready. 20 20


## Cell 2 — 读取带权重 & 名次的共现矩阵（直接用你刚写出的 parquet）

In [24]:
import glob

def pqt_to_weighted_and_rank(paths):
    from collections import defaultdict
    weights, ranks = defaultdict(dict), defaultdict(dict)
    for p in paths:
        df = pd.read_parquet(p, columns=['aid_x','aid_y','wgt'])
        df = df.sort_values(['aid_x','wgt'], ascending=[True, False])
        df['rank'] = df.groupby('aid_x').cumcount().astype('int32')
        for row in df.itertuples(index=False):
            # row.aid_x, row.aid_y, row.wgt, row.rank 都是数值字段，不会与方法名冲突
            ax, ay = int(row.aid_x), int(row.aid_y)
            weights[ax][ay] = float(row.wgt)
            ranks[ax][ay]   = int(row.rank)
    return dict(weights), dict(ranks)


# 自动发现当前工作目录下的共现产物（不用依赖 DISK_PIECES）
CLICK_PQTS = sorted(glob.glob("top_20_clicks_v*_*.pqt"))
BUYS_PQTS  = sorted(glob.glob("top_15_carts_orders_v*_*.pqt"))
B2B_PQTS   = sorted(glob.glob("top_15_buy2buy_v*_*.pqt"))
print("Found:", len(CLICK_PQTS), "clicks;", len(BUYS_PQTS), "carts_orders;", len(B2B_PQTS), "buy2buy")

click_w, click_r = pqt_to_weighted_and_rank(CLICK_PQTS)
buys_w,  buys_r  = pqt_to_weighted_and_rank(BUYS_PQTS)
b2b_w,   b2b_r   = pqt_to_weighted_and_rank(B2B_PQTS)


Found: 4 clicks; 4 carts_orders; 1 buy2buy


## Cell 3 — 候选与特征（复用你已有逻辑，最小实现）

In [25]:
from collections import Counter
import itertools

TYPE_W = {0:1, 1:6, 2:3}

def build_session_seeds(df_hist):
    aids  = df_hist.aid.tolist()
    types = df_hist.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1]))
    unique_buys = list(dict.fromkeys(df_hist[df_hist['type'].isin([1,2])].aid.tolist()[::-1]))
    return aids, types, unique_aids, unique_buys

def recent_type_score(aids, types, lo=0.1, hi=1.0):
    w = np.logspace(lo, hi, len(aids), base=2, endpoint=True) - 1.0
    sc = Counter()
    for aid, ww, t in zip(aids, w, types):
        sc[aid] += float(ww) * TYPE_W[int(t)]
    return sc

def aggregate_neighbors(unique_aids, unique_buys):
    agg_click, agg_buys, agg_b2b = Counter(), Counter(), Counter()
    for a in unique_aids:
        for b,w in click_w.get(a, {}).items(): agg_click[b] += w
        for b,w in buys_w.get(a,  {}).items(): agg_buys[b]  += w
    for a in unique_buys:
        for b,w in b2b_w.get(a,   {}).items(): agg_b2b[b]   += w
    return agg_click, agg_buys, agg_b2b

def build_candidates(df_hist, k_click=120, k_buys=120):
    aids, types, unique_aids, unique_buys = build_session_seeds(df_hist)
    agg_click, agg_buys, agg_b2b = aggregate_neighbors(unique_aids, unique_buys)
    cand_clicks = set(unique_aids) | set([b for b,_ in agg_click.most_common(k_click)])
    cand_buys   = set(unique_aids) | set([b for b,_ in (agg_buys+agg_b2b).most_common(k_buys)])
    return (aids, types, unique_aids, unique_buys, agg_click, agg_buys, agg_b2b, cand_clicks, cand_buys)

def best_rank_over_seeds(c, seeds, ranks, default=9999):
    best = default
    for s in seeds:
        r = ranks.get(s, {}).get(c, default)
        if r < best: best = r
    return best

def make_features_for_head(df_hist, head, candidates,
                           aids, types, unique_aids, unique_buys,
                           agg_click, agg_buys, agg_b2b):
    rec_sc  = recent_type_score(aids, types)
    sesslen = len(aids)
    last_ts = int(df_hist.ts.max()) if sesslen else 0
    rows, keys = [], []
    for c in candidates:
        in_hist = 1 if c in rec_sc else 0
        w_click = float(agg_click.get(c, 0.0))
        w_buys  = float(agg_buys.get(c, 0.0))
        w_b2b   = float(agg_buys.get(c, 0.0))  # 也可以用 agg_buys+agg_b2b 合并后的
        r_click = best_rank_over_seeds(c, unique_aids, click_r)
        r_buys  = best_rank_over_seeds(c, unique_aids, buys_r)
        r_b2b   = best_rank_over_seeds(c, unique_buys, b2b_r)
        f_best_click = 1.0/(1.0 + r_click)
        f_best_buys  = 1.0/(1.0 + r_buys)
        f_best_b2b   = 1.0/(1.0 + r_b2b)
        f_rec_sc = float(rec_sc.get(c, 0.0))
        if c in df_hist.aid.values:
            gap = last_ts - int(df_hist[df_hist.aid==c].ts.max())
        else:
            gap = 10*24*3600
        is_top_click = int(c in top_clicks_set)
        is_top_order = int(c in top_orders_set)
        if head=='clicks':
            rows.append([in_hist, f_rec_sc, w_click, f_best_click,
                         w_buys, f_best_buys, f_best_b2b,
                         sesslen, gap, is_top_click, is_top_order])
        else:
            rows.append([in_hist, f_rec_sc, w_buys, f_best_buys,
                         w_click, f_best_click, f_best_b2b,
                         sesslen, gap, is_top_click, is_top_order])
        keys.append(c)
    X = pd.DataFrame(rows, columns=[
        'in_hist','rec_rule',
        'w_main','best_main',
        'w_aux1','best_aux1',
        'best_b2b',  # 简化：只用名次特征，足够稳定
        'sess_len','gap',
        'is_top_click','is_top_order'
    ])
    return X, keys


## Cell 4 — 轻量训练集（会话最后 24h 当“未来”）& 训练模型

In [26]:
from tqdm import tqdm
import glob, random

def build_train_rows_from_session(df_sess):
    if df_sess.empty: return [], []
    split_ts = int(df_sess.ts.max()) - 24*60*60
    hist = df_sess[df_sess.ts <  split_ts]
    fut  = df_sess[df_sess.ts >= split_ts]
    if hist.empty or fut.empty: return [], []
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     cand_clicks, cand_buys) = build_candidates(hist)
    labs_clicks = set(fut.loc[fut['type']==0,'aid'].tolist())
    labs_buys   = set(fut.loc[fut['type'].isin([1,2]),'aid'].tolist())
    Xc, kc = make_features_for_head(hist, 'clicks', cand_clicks,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    yc = [1 if a in labs_clicks else 0 for a in kc]
    Xb, kb = make_features_for_head(hist, 'buys', cand_buys,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    yb = [1 if a in labs_buys else 0 for a in kb]
    return [(Xc, yc)], [(Xb, yb)]

def build_training_dataset(train_glob="../input/otto-chunk-data-inparquet-format/*_parquet/*",
                           max_sessions=120000, seed=42):
    files = glob.glob(train_glob)
    random.seed(seed); random.shuffle(files)
    Xc_list, yc_list, Xb_list, yb_list = [], [], [], []
    seen = 0
    with tqdm(total=max_sessions, desc="Build train set") as pbar:
        for f in files:
            df = pd.read_parquet(f)
            df['ts'] = (df['ts']/1000).astype('int32') if df['ts'].max()>1e10 else df['ts'].astype('int32')
            if df['type'].dtype=='O':
                df['type'] = df['type'].map({'clicks':0,'carts':1,'orders':2}).astype('int8')
            for _, g in df.groupby('session'):
                xc, xb = build_train_rows_from_session(g)
                for Xc,yc in xc: Xc_list.append(Xc); yc_list += yc
                for Xb,yb in xb: Xb_list.append(Xb); yb_list += yb
                seen += 1; pbar.update(1)
                if seen >= max_sessions: break
            if seen >= max_sessions: break
    Xc = pd.concat(Xc_list, ignore_index=True) if Xc_list else pd.DataFrame()
    Xb = pd.concat(Xb_list, ignore_index=True) if Xb_list else pd.DataFrame()
    return Xc, np.array(yc_list, np.int8), Xb, np.array(yb_list, np.int8)

Xc, yc, Xb, yb = build_training_dataset(max_sessions=100000)  # 可调小试跑：比如 50_000
print("Clicks train:", Xc.shape, "pos rate:", float(yc.mean()) if len(yc) else None)
print("Buys   train:", Xb.shape, "pos rate:", float(yb.mean()) if len(yb) else None)

clf_clicks = xgb.XGBClassifier(
    n_estimators=300, max_depth=7, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9,
    reg_alpha=1e-3, reg_lambda=1.0,
    tree_method='hist', random_state=2025, n_jobs=-1
)
clf_buys = xgb.XGBClassifier(
    n_estimators=300, max_depth=7, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9,
    reg_alpha=1e-3, reg_lambda=1.0,
    tree_method='hist', random_state=2025, n_jobs=-1
)

if (not Xc.empty) and (len(np.unique(yc))>1):
    clf_clicks.fit(Xc, yc)
if (not Xb.empty) and (len(np.unique(yb))>1):
    clf_buys.fit(Xb, yb)


Build train set: 100%|██████████| 100000/100000 [20:03<00:00, 83.11it/s] 


Clicks train: (5064648, 11) pos rate: 0.00850602055661124
Buys   train: (4605820, 11) pos rate: 0.001235827713631883


## Cell 5 — ML 排序函数（失败则自动回退到你原先的规则）

In [27]:
def rank_and_take20(model, X, keys, fallback_sorted):
    if model is None or X.empty:
        return fallback_sorted[:20]
    s = model.predict_proba(X)[:,1] if hasattr(model, "predict_proba") else model.predict(X)
    order = np.argsort(-s)
    ranked = [keys[i] for i in order]
    out, used = [], set()
    for a in ranked + fallback_sorted:  # 兜底不丢
        if a not in used:
            out.append(a); used.add(a)
        if len(out)==20: break
    return out

def suggest_clicks_ml(df):
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     cand_clicks, _) = build_candidates(df)
    Xc, kc = make_features_for_head(df, 'clicks', cand_clicks,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    fallback = suggest_clicks(df)  # 复用你现有的规则函数
    return rank_and_take20(clf_clicks if 'clf_clicks' in globals() else None, Xc, kc, fallback)

def suggest_buys_ml(df):
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     _, cand_buys) = build_candidates(df)
    Xb, kb = make_features_for_head(df, 'buys', cand_buys,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    fallback = suggest_buys(df)
    return rank_and_take20(clf_buys if 'clf_buys' in globals() else None, Xb, kb, fallback)


## Cell 6 — 批量特征生成 + 一次性模型推理

In [31]:
import numpy as np
import pandas as pd
from collections import defaultdict
import time, psutil

def _rank_take20_with_fallback(scores_keys_per_sess, fallback_per_sess):
    """
    scores_keys_per_sess: dict[session] -> list[(score, aid)]
    fallback_per_sess   : dict[session] -> list[aid]  (规则版候选顺序)
    return: dict[session] -> list[20 aids]
    """
    out = {}
    for sid, pairs in scores_keys_per_sess.items():
        pairs.sort(key=lambda x: -x[0])  # score 降序
        used, res = set(), []
        for _, aid in pairs:
            if aid not in used:
                res.append(aid); used.add(aid)
            if len(res) == 20: break
        # 兜底：补满 20
        if len(res) < 20:
            for aid in fallback_per_sess.get(sid, []):
                if aid not in used:
                    res.append(aid); used.add(aid)
                if len(res) == 20: break
        out[sid] = res
    return out

def _batch_predict_head(head, df_sorted, batch_sessions=120000):
    """
    head: 'clicks' or 'buys'
    df_sorted: test_df 已按 ["session","ts"] 排好
    batch_sessions: 每批处理 session 数
    """
    assert head in ("clicks", "buys")
    model = clf_clicks if head == "clicks" else clf_buys
    use_model = (model is not None) and hasattr(model, "predict_proba")

    preds_all = {}
    sessions = df_sorted["session"].drop_duplicates().values
    N = len(sessions)

    print(f"\n🚀 Start rerank inference for [{head}] on {N:,} sessions, batch size {batch_sessions}")
    t0_total = time.time()

    for start in range(0, N, batch_sessions):
        end = min(N, start + batch_sessions)
        sess_batch = set(sessions[start:end])
        batch_id = start // batch_sessions + 1

        t0 = time.time()
        print(f"\n🧩 Batch {batch_id}: sessions {start:,} ~ {end-1:,} "
              f"({end-start:,} total) | {head.upper()} head")

        g = df_sorted[df_sorted["session"].isin(sess_batch)].groupby("session", sort=False)

        X_rows, key_rows, fallback = [], [], {}

        # Step 1: 构建候选 + 特征
        for sid, df_sess in g:
            (aids, types, unique_aids, unique_buys,
             agg_click, agg_buys, agg_b2b,
             cand_clicks, cand_buys) = build_candidates(df_sess)

            if head == "clicks":
                X, keys = make_features_for_head(
                    df_sess, "clicks", cand_clicks,
                    aids, types, unique_aids, unique_buys,
                    agg_click, agg_buys, agg_b2b
                )
                fallback[sid] = suggest_clicks(df_sess)
            else:
                X, keys = make_features_for_head(
                    df_sess, "buys", cand_buys,
                    aids, types, unique_aids, unique_buys,
                    agg_click, agg_buys, agg_b2b
                )
                fallback[sid] = suggest_buys(df_sess)

            if not X.empty:
                X_rows.append(X)
                key_rows.extend([(sid, a) for a in keys])

        # Step 2: 模型批量推理
        if not X_rows:
            print("⚠️  No features in this batch, skipping.")
            continue

        X_all = pd.concat(X_rows, ignore_index=True)
        if use_model:
            print(f"🔮  Running model on {len(X_all):,} feature rows …")
            scores = model.predict_proba(X_all)[:, 1]
        else:
            print("⚙️  Model not found, using fallback only.")
            scores = np.zeros(len(X_all), dtype=np.float32)

        # Step 3: 汇总结果
        scores_per_sess = defaultdict(list)
        for (sid, aid), sc in zip(key_rows, scores):
            scores_per_sess[sid].append((float(sc), int(aid)))

        batch_pred = _rank_take20_with_fallback(scores_per_sess, fallback)
        preds_all.update(batch_pred)

        # Step 4: 打印批次耗时与内存
        used_mem = psutil.virtual_memory().used / 1024**3
        print(f"✅ Batch {batch_id} done in {time.time()-t0:.1f}s | "
              f"Mem used: {used_mem:.2f} GB")

        # 手动清理
        del X_rows, key_rows, X_all, scores_per_sess, batch_pred

    print(f"\n🏁 Finished [{head}] rerank: total {time.time()-t0_total:.1f}s")
    return preds_all


## 分别批量预测 clicks/buys → 组装提交（一次性输出）

In [ ]:
%%time
# 先对 test_df 排序一次即可
test_sorted = test_df.sort_values(["session","ts"], kind="mergesort")

# 分别批量推理两个头
clicks_top20 = _batch_predict_head("clicks", test_sorted, batch_sessions=120000)
buys_top20   = _batch_predict_head("buys",   test_sorted, batch_sessions=120000)

# 组装为提交 DataFrame
def _to_submission_block(d, suffix):
    # d: dict[session] -> list[20 aids]
    s = pd.Series({f"{sid}_{suffix}": " ".join(map(str, aids)) for sid, aids in d.items()})
    df = s.rename_axis("session_type").reset_index(name="labels")
    return df

clicks_pred_df = _to_submission_block(clicks_top20, "clicks")
orders_pred_df = _to_submission_block(buys_top20,   "orders")
carts_pred_df  = _to_submission_block(buys_top20,   "carts")

pred_df_ml = pd.concat([clicks_pred_df, orders_pred_df, carts_pred_df], ignore_index=True)
pred_df_ml.to_csv("submission_ml.csv", index=False)

# 快速自检
ok_ratio = (pred_df_ml["labels"].str.split().map(len) == 20).mean()
print("submission_ml.csv ready. 20-per-line ratio:", f"{ok_ratio:.3f}")
pred_df_ml.head()



🚀 Start rerank inference for [clicks] on 1,671,803 sessions, batch size 120000

🧩 Batch 1: sessions 0 ~ 119,999 (120,000 total) | CLICKS head
🔮  Running model on 5,113,856 feature rows …
✅ Batch 1 done in 503.0s | Mem used: 22.33 GB

🧩 Batch 2: sessions 120,000 ~ 239,999 (120,000 total) | CLICKS head
🔮  Running model on 4,920,257 feature rows …
✅ Batch 2 done in 484.5s | Mem used: 22.32 GB

🧩 Batch 3: sessions 240,000 ~ 359,999 (120,000 total) | CLICKS head
🔮  Running model on 4,834,535 feature rows …
✅ Batch 3 done in 484.7s | Mem used: 22.35 GB

🧩 Batch 4: sessions 360,000 ~ 479,999 (120,000 total) | CLICKS head
🔮  Running model on 4,794,690 feature rows …
✅ Batch 4 done in 471.0s | Mem used: 22.39 GB

🧩 Batch 5: sessions 480,000 ~ 599,999 (120,000 total) | CLICKS head
🔮  Running model on 4,851,898 feature rows …
✅ Batch 5 done in 476.9s | Mem used: 22.47 GB

🧩 Batch 6: sessions 600,000 ~ 719,999 (120,000 total) | CLICKS head
🔮  Running model on 4,811,734 feature rows …
✅ Batch 6 do

## Cell 7 — 快速自检（每行 20 个、无空值）

In [ ]:
# 简单检查
n_rows = pred_df_ml.shape[0]
ok_len = (pred_df_ml['labels'].str.split().map(len) == 20).mean()
print(f"Rows: {n_rows}, 20-per-line ratio: {ok_len:.3f}")
# 两份提交都在：submission_rules.csv（基线） + submission_ml.csv（ML）
